# NumCompute-Stream — Streaming ML Demo

This notebook demonstrates the full streaming pipeline:
1. Load a dataset from CSV using NumPy I/O
2. Split into chunks to simulate streaming
3. Train incrementally with `.partial_fit()` chunk by chunk
4. Log and visualise metrics over time
5. Compare single tree vs Random Forest

## 0. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import RandomForestClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.metrics import AccuracyMetric, F1Metric, RollingAccuracy
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
    plot_confusion_matrix,
)
from numcompute_stream.stats import StreamStats

print('All imports OK')

## 1. Generate & Save a Synthetic Dataset as CSV

We generate a binary classification dataset and save it as CSV,
then reload it using `np.genfromtxt` to simulate a real I/O pipeline.

In [ ]:
# Generate synthetic dataset
rng = np.random.default_rng(42)
n_samples, n_features = 1000, 6

X = rng.standard_normal((n_samples, n_features))
# Non-linear boundary: sum of first two features + interaction
y = ((X[:, 0] + X[:, 1] + 0.5 * X[:, 0] * X[:, 2]) > 0).astype(int)

# Add 5% label noise to make it realistic
noise_idx = rng.choice(n_samples, size=int(0.05 * n_samples), replace=False)
y[noise_idx] = 1 - y[noise_idx]

# Save as CSV
os.makedirs('data', exist_ok=True)
csv_path = 'data/stream_dataset.csv'
header = ','.join([f'feature_{i}' for i in range(n_features)] + ['label'])
np.savetxt(csv_path, np.column_stack([X, y]), delimiter=',', header=header, comments='')

print(f'Saved {n_samples} samples to {csv_path}')
print(f'Class distribution: {np.bincount(y)}')

In [ ]:
# Reload from CSV — simulating real I/O pipeline
data = np.genfromtxt(csv_path, delimiter=',', skip_header=1)
X_loaded = data[:, :-1]
y_loaded = data[:, -1].astype(int)

print(f'Loaded shape: X={X_loaded.shape}, y={y_loaded.shape}')
print(f'Features: {X_loaded.shape[1]}, Classes: {np.unique(y_loaded)}')

## 2. Split into Chunks — Simulate Streaming

In [ ]:
N_CHUNKS = 10

chunks = np.array_split(np.arange(n_samples), N_CHUNKS)
chunks_X = [X_loaded[idx] for idx in chunks]
chunks_y = [y_loaded[idx] for idx in chunks]

print(f'Split into {N_CHUNKS} chunks:')
for i, (cx, cy) in enumerate(zip(chunks_X, chunks_y)):
    print(f'  Chunk {i+1:2d}: {len(cy):4d} samples  |  class balance: {cy.mean():.2f}')

## 3. Streaming Stats — Monitor Data Distribution

In [ ]:
ss = StreamStats()
for cx in chunks_X:
    ss.update_stats(cx)

stats = ss.get_stats()
print('Running statistics after all chunks:')
print(f"  Mean per feature:     {stats['mean'].round(3)}")
print(f"  Std  per feature:     {stats['std'].round(3)}")
print(f"  Min  per feature:     {stats['min'].round(3)}")
print(f"  Max  per feature:     {stats['max'].round(3)}")
print(f"  Total samples seen:   {stats['n_samples_seen']}")

## 4. Build Streaming Pipelines

In [ ]:
# Single Decision Tree pipeline
pipe_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  DecisionTreeClassifier(max_depth=5, random_state=0)),
])

# Random Forest pipeline
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=10, max_depth=5, random_state=0)),
])

# StreamTrainers with metrics
trainer_dt = StreamTrainer(
    pipeline=pipe_dt,
    metrics={'f1': F1Metric(), 'rolling_acc': RollingAccuracy(window=3)},
    verbose=True,
)

trainer_rf = StreamTrainer(
    pipeline=pipe_rf,
    metrics={'f1': F1Metric(), 'rolling_acc': RollingAccuracy(window=3)},
    verbose=True,
)

print('Pipelines ready.')

## 5. Incremental Training — Chunk by Chunk

In [ ]:
print('=== Decision Tree Streaming ===')
logs_dt = trainer_dt.run(chunks_X, chunks_y)
trainer_dt.summary()

In [ ]:
print('\n=== Random Forest Streaming ===')
logs_rf = trainer_rf.run(chunks_X, chunks_y)
trainer_rf.summary()

## 6. Visualise Metrics Over Time

In [ ]:
# Cumulative accuracy over chunks
fig = plot_metric_over_time(
    logs_dt['cumulative_accuracy'],
    title='Decision Tree — Cumulative Accuracy over Chunks',
    ylabel='Accuracy',
    show=True,
)
plt.show()

In [ ]:
# F1 score over chunks
fig = plot_metric_over_time(
    logs_dt['f1'],
    title='Decision Tree — F1 Score over Chunks',
    ylabel='F1 (macro)',
    color='#EA580C',
    show=True,
)
plt.show()

In [ ]:
# Rolling accuracy (last 3 chunks)
fig = plot_metric_over_time(
    logs_dt['rolling_acc'],
    title='Decision Tree — Rolling Accuracy (window=3)',
    ylabel='Rolling Accuracy',
    color='#16A34A',
    show=True,
)
plt.show()

## 7. Compare Decision Tree vs Random Forest

In [ ]:
fig = compare_models(
    logs_dt['cumulative_accuracy'],
    logs_rf['cumulative_accuracy'],
    labels=('Decision Tree', 'Random Forest'),
    title='Cumulative Accuracy: Decision Tree vs Random Forest',
    ylabel='Accuracy',
    show=True,
)
plt.show()

In [ ]:
fig = compare_models(
    logs_dt['f1'],
    logs_rf['f1'],
    labels=('Decision Tree', 'Random Forest'),
    title='F1 Score: Decision Tree vs Random Forest',
    ylabel='F1 (macro)',
    show=True,
)
plt.show()

## 8. Predictions vs Ground Truth (Latest Chunk)

In [ ]:
# Use last chunk for visualisation
X_last = chunks_X[-1]
y_last = chunks_y[-1]

y_pred_dt = pipe_dt.predict(X_last)
y_pred_rf = pipe_rf.predict(X_last)

print(f'Last chunk size: {len(y_last)}')
print(f'DT  accuracy on last chunk: {(y_pred_dt == y_last).mean():.4f}')
print(f'RF  accuracy on last chunk: {(y_pred_rf == y_last).mean():.4f}')

In [ ]:
fig = plot_predictions_vs_ground_truth(
    y_last, y_pred_rf,
    title='Random Forest — Predictions vs Ground Truth (Last Chunk)',
    show=True,
)
plt.show()

## 9. Confusion Matrix

In [ ]:
from numcompute_stream.metrics import StreamingConfusionMatrix

cm_rf = StreamingConfusionMatrix()
for cx, cy in zip(chunks_X, chunks_y):
    cm_rf.update(cy, pipe_rf.predict(cx))

fig = plot_confusion_matrix(
    cm_rf.matrix_,
    class_labels=['Class 0', 'Class 1'],
    title='Random Forest — Accumulated Confusion Matrix',
    show=True,
)
plt.show()

## 10. Final Summary

In [ ]:
print('=' * 55)
print('  FINAL RESULTS SUMMARY')
print('=' * 55)
print(f"  Chunks processed       : {N_CHUNKS}")
print(f"  Total samples          : {n_samples}")
print(f"  Features               : {n_features}")
print()
print(f"  Decision Tree")
print(f"    Cumulative accuracy  : {logs_dt['cumulative_accuracy'][-1]:.4f}")
print(f"    Final F1             : {logs_dt['f1'][-1]:.4f}")
print(f"    Avg chunk time       : {np.mean(logs_dt['chunk_time_s'])*1000:.2f} ms")
print()
print(f"  Random Forest (10 trees)")
print(f"    Cumulative accuracy  : {logs_rf['cumulative_accuracy'][-1]:.4f}")
print(f"    Final F1             : {logs_rf['f1'][-1]:.4f}")
print(f"    Avg chunk time       : {np.mean(logs_rf['chunk_time_s'])*1000:.2f} ms")
print('=' * 55)